In [1]:
import logging
logging.basicConfig(
    # make sure debug level logs are shown
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()])


### Save paper PDF

Ensure that you can save the paper in a specific directory.

In [2]:
import arxiv_search
import os

download_dir = "test/pdfs/arxiv"
os.makedirs(download_dir, exist_ok=True)

# download attention is all you need paper: https://arxiv.org/pdf/2510.26641
arxiv_search.download_pdf("2510.26641",save_dir=download_dir)


2025-11-01 05:33:25,548 [DEBUG] Save dir:test/pdfs/arxiv
2025-11-01 05:33:25,549 [DEBUG] arXiv url: https://arxiv.org/pdf/2510.26641
2025-11-01 05:33:25,550 [DEBUG] Downloading https://arxiv.org/pdf/2510.26641 to test/pdfs/arxiv/2510.26641.pdf
2025-11-01 05:33:28,796 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:33:29,051 [DEBUG] https://arxiv.org:443 "GET /pdf/2510.26641 HTTP/1.1" 200 10325344
2025-11-01 05:33:29,502 [INFO] Saved: test/pdfs/arxiv/2510.26641.pdf


### Build a search URL

Lets use the custom function to build a search URL

In [3]:
topics = [
    "attention is all you need",
    "machine learning in the modern era"
]

urls = []

for topic in topics:
    urls.append(arxiv_search.build_arxiv_search_url(query=topic, size=20))

print(urls)

['https://arxiv.org/search/cs?query=attention+is+all+you+need&searchtype=all&abstracts=show&order=-announced_date_first&size=25', 'https://arxiv.org/search/cs?query=machine+learning+in+the+modern+era&searchtype=all&abstracts=show&order=-announced_date_first&size=25']


### Scrap 

Scrap data from the urls in the background and save to json file

In [4]:
import time
# Prepare directory
scraped_directory = "test/scraped"
os.makedirs(scraped_directory, exist_ok=True)

# output file
output_file = []

# run the scraper for each url
for index, url in enumerate(urls):
    file_location = f"{scraped_directory}/{index}_scraped.json"
    arxiv_search.run_scraper_in_background(url=url, output_file=file_location)
    time.sleep(3)

    output_file.append(file_location)

print(output_file)

2025-11-01 05:33:29,542 [DEBUG] Scraping started in background. Results will be saved to test/scraped/0_scraped.json.


2025-11-01 05:33:32,543 [DEBUG] Scraping started in background. Results will be saved to test/scraped/1_scraped.json.


['test/scraped/0_scraped.json', 'test/scraped/1_scraped.json']


Scrape metadata from the json file

In [5]:
print(output_file)

['test/scraped/0_scraped.json', 'test/scraped/1_scraped.json']


In [6]:
import logging

pages_metadata = []

for index, value in enumerate[str](output_file):
    logging.debug(f"{index}.{value}")
    # extend the list (we only need one list)
    pages_metadata.extend(arxiv_search.scrape_arxiv_details_from_json_threaded(value))

print(len(pages_metadata))

2025-11-01 05:33:35,559 [DEBUG] 0.test/scraped/0_scraped.json
2025-11-01 05:33:36,337 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:33:36,641 [DEBUG] https://arxiv.org:443 "GET /search/cs?query=machine+learning+in+the+modern+era&searchtype=all&abstracts=show&order=-announced_date_first&size=25 HTTP/1.1" 200 141870
2025-11-01 05:33:36,716 [DEBUG] Saved 25 papers to test/scraped/1_scraped.json
2025-11-01 05:33:36,720 [DEBUG] Type of output file:<class 'str'>
2025-11-01 05:33:39,503 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:33:39,895 [DEBUG] https://arxiv.org:443 "GET /search/cs?query=attention+is+all+you+need&searchtype=all&abstracts=show&order=-announced_date_first&size=25 HTTP/1.1" 200 138390
2025-11-01 05:33:39,984 [DEBUG] Saved 25 papers to test/scraped/0_scraped.json
2025-11-01 05:33:39,985 [DEBUG] Type of output file:<class 'str'>
2025-11-01 05:33:40,992 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:33:4

125


In [7]:
pages_metadata

[{'url': 'https://arxiv.org/abs/2510.18870',
  'title': 'Triangle Multiplication Is All You Need For Biomolecular Structure Representations',
  'abstract': 'AlphaFold has transformed protein structure prediction, but emerging applications such as virtual ligand screening, proteome-wide folding, and de novo binder design demand predictions at a massive scale, where runtime and memory costs become prohibitive. A major bottleneck lies in the Pairformer backbone of AlphaFold3-style models, which relies on computationally expensive triangular primitives-especially triangle attention-for pairwise reasoning. We introduce Pairmixer, a streamlined alternative that eliminates triangle attention while preserving higher-order geometric reasoning capabilities that are critical for structure prediction. Pairmixer substantially improves computational efficiency, matching state-of-the-art structure predictors across folding and docking benchmarks, delivering up to 4x faster inference on long sequences

In [9]:
# save the combined list to a file
# enriched data
enriched_data_dir = "test/enriched"
os.makedirs(enriched_data_dir, exist_ok=True)

saved_files = []

if isinstance(pages_metadata[0], list):
    for index, page_items in enumerate(pages_metadata):
        file_location = f"{enriched_data_dir}/{index}_papers_metadata.json"
        saved_files.append(file_location)
        arxiv_search.save_arxiv_scraped_details(results=page_items, output_file=file_location)
else:
    file_location = f"{enriched_data_dir}/papers_metadata.json"
    saved_files.append(file_location)
    arxiv_search.save_arxiv_scraped_details(results=pages_metadata, output_file=file_location)


  


saved_files
# metadata_file = f"{enriched_data_dir}/papers_metadata.json"
# arxiv_search.save_arxiv_scraped_details(results=pages_metadata, output_file=metadata_file)

2025-11-01 05:53:15,316 [DEBUG] First item in list: {'url': 'https://arxiv.org/abs/2510.18870', 'title': 'Triangle Multiplication Is All You Need For Biomolecular Structure Representations', 'abstract': 'AlphaFold has transformed protein structure prediction, but emerging applications such as virtual ligand screening, proteome-wide folding, and de novo binder design demand predictions at a massive scale, where runtime and memory costs become prohibitive. A major bottleneck lies in the Pairformer backbone of AlphaFold3-style models, which relies on computationally expensive triangular primitives-especially triangle attention-for pairwise reasoning. We introduce Pairmixer, a streamlined alternative that eliminates triangle attention while preserving higher-order geometric reasoning capabilities that are critical for structure prediction. Pairmixer substantially improves computational efficiency, matching state-of-the-art structure predictors across folding and docking benchmarks, deliver

['test/enriched/papers_metadata.json']

In [ ]:
import arxiv_search
print(arxiv_search.get_pdf_arxiv)

for index, file in enumerate(saved_files):
    saved_dir = f"test/pdfs/arxiv/{index}"
    arxiv_search.get_pdf_arxiv(cleaned_json=file, save_dir=saved_dir)

2025-11-01 05:55:06,972 [DEBUG] Threadsafe pdf arxiv
2025-11-01 05:55:06,973 [INFO] Saving to test/pdfs/arxiv/0
2025-11-01 05:55:06,975 [DEBUG] Save dir:test/pdfs/arxiv/0
2025-11-01 05:55:06,976 [DEBUG] arXiv url: https://arxiv.org/pdf/2510.18870
2025-11-01 05:55:06,977 [DEBUG] Downloading https://arxiv.org/pdf/2510.18870 to test/pdfs/arxiv/0/2510.18870.pdf
2025-11-01 05:55:06,976 [DEBUG] Started thread for 2510.18870
2025-11-01 05:55:06,977 [DEBUG] Save dir:test/pdfs/arxiv/0
2025-11-01 05:55:06,978 [DEBUG] arXiv url: https://arxiv.org/pdf/2510.26641
2025-11-01 05:55:06,978 [DEBUG] Started thread for 2510.26641
2025-11-01 05:55:06,979 [DEBUG] Save dir:test/pdfs/arxiv/0
2025-11-01 05:55:06,979 [DEBUG] Downloading https://arxiv.org/pdf/2510.26641 to test/pdfs/arxiv/0/2510.26641.pdf
2025-11-01 05:55:06,979 [DEBUG] Started thread for 2510.19861
2025-11-01 05:55:06,980 [DEBUG] arXiv url: https://arxiv.org/pdf/2510.19861
2025-11-01 05:55:06,981 [DEBUG] Downloading https://arxiv.org/pdf/2510.

<function get_pdf_arxiv at 0xffff91d74fe0>


2025-11-01 05:55:07,172 [DEBUG] Started thread for 2210.02493
2025-11-01 05:55:07,172 [DEBUG] arXiv url: https://arxiv.org/pdf/2211.04346
2025-11-01 05:55:07,174 [DEBUG] arXiv url: https://arxiv.org/pdf/2210.02493
2025-11-01 05:55:07,176 [DEBUG] Downloading https://arxiv.org/pdf/2210.02493 to test/pdfs/arxiv/0/2210.02493.pdf
2025-11-01 05:55:07,176 [DEBUG] Save dir:test/pdfs/arxiv/0
2025-11-01 05:55:07,179 [DEBUG] arXiv url: https://arxiv.org/pdf/2207.03782
2025-11-01 05:55:07,180 [DEBUG] Downloading https://arxiv.org/pdf/2207.03782 to test/pdfs/arxiv/0/2207.03782.pdf
2025-11-01 05:55:07,176 [DEBUG] Started thread for 2207.03782
2025-11-01 05:55:07,176 [DEBUG] Downloading https://arxiv.org/pdf/2211.04346 to test/pdfs/arxiv/0/2211.04346.pdf
2025-11-01 05:55:07,181 [DEBUG] Started thread for 2207.01903
2025-11-01 05:55:07,181 [DEBUG] Save dir:test/pdfs/arxiv/0
2025-11-01 05:55:07,182 [DEBUG] Save dir:test/pdfs/arxiv/0
2025-11-01 05:55:07,183 [DEBUG] arXiv url: https://arxiv.org/pdf/2205.

2025-11-01 05:55:10,223 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:55:10,341 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:55:10,451 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:55:10,453 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:55:10,466 [DEBUG] https://arxiv.org:443 "GET /pdf/2406.03470 HTTP/1.1" 200 2020482
2025-11-01 05:55:10,491 [DEBUG] https://arxiv.org:443 "GET /pdf/2510.20795 HTTP/1.1" 200 2052095
2025-11-01 05:55:10,580 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:55:10,610 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-01 05:55:10,622 [INFO] Saved: test/pdfs/arxiv/0/2406.03470.pdf
2025-11-01 05:55:10,633 [DEBUG] https://arxiv.org:443 "GET /pdf/2509.06383 HTTP/1.1" 200 557876
2025-11-01 05:55:10,644 [INFO] Saved: test/pdfs/arxiv/0/2510.20795.pdf
2025-11-01 05:55:10,720 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2